In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/rentora_final_week1.csv', low_memory=False)
df.shape

(117443, 14)

In [3]:
df.dtypes

city                 str
locality             str
bhk              float64
rent             float64
size_sqft            str
furnishing           str
bathrooms        float64
latitude         float64
longitude        float64
source               str
near_highway        bool
near_mall           bool
near_river          bool
near_mountain       bool
dtype: object

In [4]:
#find bad size_sqft rows
numeric_sqft = pd.to_numeric(df['size_sqft'], errors='coerce') # meaning of errors='coerce'-If you can't convert it to a number, turn it into NaN.
bad_rows = df[numeric_sqft.isna()]
print('Rows where size_sqft is not a clean number:', len(bad_rows))
bad_rows[['city','locality','size_sqft']].head(20)

Rows where size_sqft is not a clean number: 11137


,city,locality,size_sqft
106306,Delhi,Kalkaji,400 sq ft
106307,Delhi,Mansarover Garden,400 sq ft
106308,Delhi,Uttam Nagar,500 sq ft
106309,Delhi,Model Town,"1,020 sq ft"
106310,Delhi,Sector 13 Rohini,810 sq ft
106311,Delhi,DLF Farms,750 sq ft
106312,Delhi,laxmi nagar,"1,300 sq ft"
106313,Delhi,Swasthya Vihar,"1,200 sq ft"
106314,Delhi,Janakpuri,"1,100 sq ft"
106315,Delhi,Pitampura,"2,500 sq ft"


In [5]:
#clean size_sqft
df['size_sqft'] = (
    df['size_sqft']
    .astype(str)
    .str.replace('sq ft', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['size_sqft'] = pd.to_numeric(df['size_sqft'], errors='coerce')

print('Nulls after conversion:', df['size_sqft'].isnull().sum())
df['size_sqft'].describe()

Nulls after conversion: 0


count    117443.000000
mean       1323.881985
std        1153.872610
min           3.000000
25%         664.500000
50%        1000.000000
75%        1480.000000
max       15500.000000
Name: size_sqft, dtype: float64

In [6]:
#Skewness = "Is my data leaning too much to one side?"
#rent skew check
print('Skewness:', df['rent'].skew())
df['rent'].describe()


Skewness: 5.172343473382728


count    1.174430e+05
mean     5.061817e+04
std      9.887811e+04
min      1.200000e+03
25%      1.300000e+04
50%      2.200000e+04
75%      4.000000e+04
max      1.282000e+06
Name: rent, dtype: float64

In [7]:
#log transform check
#Log transformation = "Compress the huge values so they don't dominate."
log_rent_skew = np.log1p(df['rent']).skew()
print('Skewness after log1p transform:', log_rent_skew)

Skewness after log1p transform: 1.0544172660786453


In [8]:
#city counts
df['city'].value_counts()

city
Mumbai       37298
Delhi        23440
Pune         17329
Bangalore    13396
Ahmedabad     7676
Kolkata       6425
Chennai       6101
Hyderabad     5772
Hisar            6
Name: count, dtype: int64

In [9]:
#geo-feature rate by city
df.groupby('city')[['near_highway','near_mall','near_river','near_mountain']].mean()

,near_highway,near_mall,near_river,near_mountain
city,,,,
Ahmedabad,0.804716,0.340151,0.941506,0.000000
Bangalore,0.985742,0.290609,0.994924,0.054867
Chennai,0.975086,0.268317,0.979675,0.017866
Delhi,0.909556,0.317022,0.964804,0.008959
Hisar,0.000000,0.000000,0.000000,0.000000
Hyderabad,0.967602,0.572938,0.976438,0.039674
Kolkata,0.953930,0.411984,0.974008,0.000000
Mumbai,0.979168,0.311169,0.945708,0.240254
Pune,0.974494,0.398407,0.872930,0.079578


In [10]:
#correlation with rent
corr_cols = ['rent','bhk','size_sqft','bathrooms','near_highway','near_mall','near_river','near_mountain']
corr_df = df[corr_cols].copy()
for c in ['near_highway','near_mall','near_river','near_mountain']:
    corr_df[c] = corr_df[c].astype(int)
corr_df.corr()['rent'].sort_values(ascending=False)

rent             1.000000
size_sqft        0.797314
bathrooms        0.576719
bhk              0.565886
near_river       0.048336
near_highway     0.046964
near_mall        0.030955
near_mountain   -0.015022
Name: rent, dtype: float64

Which columns are related to house rent, and how strongly are they related?

Correlation is a number between -1 and +1 that tells us how two variables move in relation to each other
| Correlation | Meaning                           |
| ----------: | --------------------------------- |
|        `+1` | Very strong positive relationship |
|      `+0.7` | Strong positive relationship      |
|      `+0.5` | Moderate positive relationship    |
|         `0` | Almost no relationship            |
|      `-0.5` | Moderate negative relationship    |
|        `-1` | Very strong negative relationship |



In [11]:
#bulk-fetch raw coordinates for all 8 cities (same as Week 1)
import requests, time, math
from tqdm import tqdm

OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter"
]
headers = {"User-Agent": "RentoraAI/1.0 (https://github.com/anisha-1811/Rentora-AI)"}

city_centers = {
    'Delhi': (28.7041, 77.1025), 'Mumbai': (19.0760, 72.8777), 'Pune': (18.5204, 73.8567),
    'Ahmedabad': (23.0225, 72.5714), 'Bangalore': (12.9716, 77.5946),
    'Chennai': (13.0827, 80.2707), 'Hyderabad': (17.3850, 78.4867), 'Kolkata': (22.5726, 88.3639)
}

def get_city_features(lat, lon, radius=30000):
    query = f"""
[out:json][timeout:90];
(
way["highway"~"trunk|primary|motorway"](around:{radius},{lat},{lon});
node["shop"="mall"](around:{radius},{lat},{lon});
way["natural"="water"](around:{radius},{lat},{lon});
node["natural"="peak"](around:{radius},{lat},{lon});
);
out center;
"""
    for url in OVERPASS_SERVERS:
        try:
            response = requests.post(url, data=query, headers=headers, timeout=90)
            if response.status_code == 200:
                return response.json().get("elements", [])
        except Exception as e:
            print(f"Error at {url}: {e}")
            continue
    return []

city_features = {}
for city, (lat, lon) in city_centers.items():
    print(f"Fetching bulk features for {city}...")
    city_features[city] = get_city_features(lat, lon)
    print(f"  {city}: {len(city_features[city])} features found")
    time.sleep(3)

Fetching bulk features for Delhi...
  Delhi: 7692 features found
Fetching bulk features for Mumbai...
  Mumbai: 6389 features found
Fetching bulk features for Pune...
  Pune: 4324 features found
Fetching bulk features for Ahmedabad...
  Ahmedabad: 2720 features found
Fetching bulk features for Bangalore...
  Bangalore: 10831 features found
Fetching bulk features for Chennai...
  Chennai: 5319 features found
Fetching bulk features for Hyderabad...
  Hyderabad: 5798 features found
Fetching bulk features for Kolkata...
  Kolkata: 11764 features found


In [12]:
# Find any city that came back empty, regardless of which one it is
failed_cities = {city: city_centers[city] for city, feats in city_features.items() if len(feats) == 0}
print(f"Cities needing retry: {list(failed_cities.keys())}")

max_attempts = 3
for attempt in range(1, max_attempts + 1):
    if not failed_cities:
        break
    print(f"\n--- Retry attempt {attempt} ---")
    still_failed = {}
    for city, (lat, lon) in failed_cities.items():
        print(f"Retrying {city}...")
        time.sleep(5)
        result = get_city_features(lat, lon)
        if len(result) > 0:
            city_features[city] = result
            print(f"  {city}: {len(result)} features found — fixed")
        else:
            print(f"  {city}: still 0 — will retry again")
            still_failed[city] = (lat, lon)
    failed_cities = still_failed

if failed_cities:
    print(f"\nStill unresolved after {max_attempts} attempts: {list(failed_cities.keys())}")
else:
    print("\nAll cities now have data.")

Cities needing retry: []

All cities now have data.


In [13]:
#extract coordinates by type (identical to Week 1)
def extract_features_by_type(elements):
    highways, malls, rivers, mountains = [], [], [], []
    for e in elements:
        tags = e.get('tags', {})
        lat = e.get('lat') or e.get('center', {}).get('lat')
        lon = e.get('lon') or e.get('center', {}).get('lon')
        if lat is None or lon is None:
            continue
        if tags.get('highway') in ['motorway', 'primary', 'trunk']:
            highways.append((lat, lon))
        if tags.get('shop') == 'mall':
            malls.append((lat, lon))
        if tags.get('natural') == 'water':
            rivers.append((lat, lon))
        if tags.get('natural') == 'peak':
            mountains.append((lat, lon))
    return highways, malls, rivers, mountains

city_feature_coords = {}
for city, elements in city_features.items():
    city_feature_coords[city] = extract_features_by_type(elements)
    h, m, r, mt = city_feature_coords[city]
    print(f"{city}: {len(h)} highways, {len(m)} malls, {len(r)} rivers, {len(mt)} mountains")

Delhi: 3734 highways, 29 malls, 2318 rivers, 3 mountains
Mumbai: 3974 highways, 23 malls, 1073 rivers, 38 mountains
Pune: 3255 highways, 15 malls, 516 rivers, 18 mountains
Ahmedabad: 1886 highways, 6 malls, 393 rivers, 0 mountains
Bangalore: 4695 highways, 14 malls, 5129 rivers, 9 mountains
Chennai: 3190 highways, 8 malls, 1171 rivers, 8 mountains
Hyderabad: 3917 highways, 78 malls, 622 rivers, 7 mountains
Kolkata: 2331 highways, 7 malls, 9186 rivers, 0 mountains


In [14]:
#save raw coordinates this time, so this never has to be re-fetched again
import json
serializable = {city: {'highways': h, 'malls': m, 'rivers': r, 'mountains': mt}
                 for city, (h, m, r, mt) in city_feature_coords.items()}
with open('../data/city_feature_coords.json', 'w') as f:
    json.dump(serializable, f)
print("Saved raw coordinates — no need to re-fetch again")

Saved raw coordinates — no need to re-fetch again


In [15]:
#minimum-distance function (swaps any(<=radius) for min(distance))
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*R*math.asin(math.sqrt(a))

def min_distances(lat, lon, city):
    if city not in city_feature_coords:
        return {"dist_highway_m": None, "dist_mall_m": None, "dist_river_m": None, "dist_mountain_m": None}
    highways, malls, rivers, mountains = city_feature_coords[city]
    def nearest(points):
        return min((haversine(lat, lon, p[0], p[1]) for p in points), default=None)
    return {
        "dist_highway_m": nearest(highways),
        "dist_mall_m": nearest(malls),
        "dist_river_m": nearest(rivers),
        "dist_mountain_m": nearest(mountains)
    }

In [16]:
#apply to every unique location, same pattern as Week 1
unique_locs = df[["city", "locality", "latitude", "longitude"]].drop_duplicates(subset=["latitude", "longitude"])

results = []
for _, row in tqdm(unique_locs.iterrows(), total=len(unique_locs)):
    dists = min_distances(row.latitude, row.longitude, row.city)
    results.append({"latitude": row.latitude, "longitude": row.longitude, **dists})

dist_df = pd.DataFrame(results)
dist_df.to_csv('../data/geo_distances_all.csv', index=False)
dist_df.describe()

100%|██████████| 7765/7765 [00:42<00:00, 182.24it/s]


,latitude,longitude,dist_highway_m,dist_mall_m,dist_river_m,dist_mountain_m
count,7765.000000,7765.000000,7.760000e+03,7.760000e+03,7.760000e+03,7.340000e+03
mean,20.231747,75.334433,6.454272e+03,9.868437e+03,6.619775e+03,1.213745e+04
std,4.556657,3.455881,1.118494e+05,1.122380e+05,1.118751e+05,1.151638e+05
min,12.746772,18.497896,0.000000e+00,1.059574e+01,8.696542e-01,0.000000e+00
25%,18.546909,72.942917,2.516757e+02,1.710322e+03,4.404827e+02,3.106023e+03
50%,19.062258,73.875305,5.503632e+02,3.229256e+03,7.614928e+02,5.208314e+03
75%,19.467953,77.209679,1.097513e+03,5.456733e+03,1.273469e+03,9.293525e+03
max,31.106586,91.792137,5.780229e+06,5.790562e+06,5.782584e+06,5.789415e+06


In [17]:
#merge distance features into the working dataframe
df = df.merge(dist_df, on=["latitude", "longitude"], how="left")
df[['dist_highway_m','dist_mall_m','dist_river_m','dist_mountain_m']].describe()

,dist_highway_m,dist_mall_m,dist_river_m,dist_mountain_m
count,1.174370e+05,1.174370e+05,1.174370e+05,1.033360e+05
mean,1.094986e+03,4.125458e+03,1.378701e+03,6.695834e+03
std,3.355563e+04,3.378399e+04,3.356261e+04,3.599219e+04
min,0.000000e+00,1.059574e+01,8.696542e-01,0.000000e+00
25%,1.418620e+02,1.551152e+03,4.237173e+02,3.311892e+03
50%,3.489346e+02,3.038094e+03,6.839412e+02,5.110079e+03
75%,7.450357e+02,5.112608e+03,1.118511e+03,9.140352e+03
max,5.780229e+06,5.790562e+06,5.782584e+06,5.789415e+06


In [18]:
# Find the worst offenders and check if their city label makes sense
suspicious = df[df['dist_highway_m'] > 50000][['city','locality','latitude','longitude','dist_highway_m']]
print(f"Rows with dist_highway_m > 50km: {len(suspicious)}")
suspicious.sort_values('dist_highway_m', ascending=False).head(15)

Rows with dist_highway_m > 50km: 62


,city,locality,latitude,longitude,dist_highway_m
115837,Pune,Wanowrie,18.497896,18.497896,5.780229e+06
115836,Pune,Wanowrie,18.497896,18.497896,5.780229e+06
114956,Pune,Ravet,18.653852,18.653852,5.761191e+06
111367,Mumbai,Borivali West,26.140644,91.792137,2.061468e+06
114055,Mumbai,Santacruz East,22.601080,88.451591,1.635215e+06
116142,Pune,Tingre Nagar,31.106586,77.161819,1.409946e+06
111662,Mumbai,Mira Road East,23.429319,85.324120,1.347137e+06
110266,Mumbai,Mulund West,30.888254,75.844421,1.317248e+06
110858,Mumbai,Mulund West,30.879341,75.850121,1.316414e+06
111596,Mumbai,Khar,26.878197,80.951424,1.167495e+06


In [19]:
missing_mountain = df[df['dist_mountain_m'].isna()]
missing_mountain['city'].value_counts()

city
Ahmedabad    7676
Kolkata      6425
Hisar           6
Name: count, dtype: int64

In [20]:
#add one new cell after cell 19, filtering both corruption patterns:
before = len(df)
df = df[df['latitude'] != df['longitude']]           # catches the 3 duplicated-value rows
df = df[df['dist_highway_m'] < 50000].copy()          # catches the wrong-city coordinate rows
print(f"Dropped {before - len(df)} rows")
df.shape

Dropped 68 rows


(117375, 18)

In [21]:
#add another new cell right after, for the mountain-distance fill:
df['dist_mountain_m'] = df['dist_mountain_m'].fillna(999999)
df['dist_mountain_m'].isna().sum()

np.int64(0)

In [22]:
#Re-check correlation with the new distance features (this was the whole point of building them — see if they're better than the old booleans):
corr_cols = ['rent','bhk','size_sqft','bathrooms','dist_highway_m','dist_mall_m','dist_river_m','dist_mountain_m']
df[corr_cols].corr()['rent'].sort_values(ascending=False)

rent               1.000000
size_sqft          0.797244
bathrooms          0.576750
bhk                0.565963
dist_highway_m    -0.011371
dist_river_m      -0.011804
dist_mall_m       -0.072567
dist_mountain_m   -0.098943
Name: rent, dtype: float64

In [23]:
# Test a range of radii, check which one produces a boolean with reasonable variance 
# AND correlates better with rent---for(dist_highway_m)
for radius in [250, 500, 750, 1000, 1500, 2000]:
    near_hw = (df['dist_highway_m'] <= radius).astype(int)
    true_rate = near_hw.mean()
    corr = near_hw.corr(df['rent'])
    print(f"radius={radius}m | True rate={true_rate:.2%} | corr with rent={corr:.4f}")

radius=250m | True rate=41.37% | corr with rent=-0.1010
radius=500m | True rate=61.67% | corr with rent=-0.0892
radius=750m | True rate=75.06% | corr with rent=-0.0512
radius=1000m | True rate=82.96% | corr with rent=-0.0296
radius=1500m | True rate=91.84% | corr with rent=0.0325
radius=2000m | True rate=95.23% | corr with rent=0.0470


The correlation flips sign as radius grows. That's a real, physically sensible pattern, not noise: being genuinely right next to a highway (within 250–750m) is a negative for rent — noise, pollution, traffic — while being generally in the vicinity of one (1500–2000m) without being adjacent is a mild positive — good connectivity, easy access, without the downsides. Real estate has exactly this kind of non-monotonic relationship all the time (e.g. "near the highway" is bad, "near the highway exit but not on it" is good).

In [24]:
# Test a range of radii, check which one produces a boolean with reasonable variance 
# AND correlates better with rent  ---for (dist_highway_m, dist_mall_m, dist_river_m, dist_mountain_m)
for radius in [250, 500, 750, 1000, 1500, 2000]:
    near_hw = (df['dist_mall_m'] <= radius).astype(int)
    true_rate = near_hw.mean()
    corr = near_hw.corr(df['rent'])
    print(f"radius={radius}m | True rate={true_rate:.2%} | corr with rent={corr:.4f}")

radius=250m | True rate=1.93% | corr with rent=-0.0328
radius=500m | True rate=4.66% | corr with rent=0.0021
radius=750m | True rate=7.15% | corr with rent=0.0231
radius=1000m | True rate=13.26% | corr with rent=0.0160
radius=1500m | True rate=23.85% | corr with rent=0.0049
radius=2000m | True rate=34.18% | corr with rent=0.0302


In [25]:
# Test a range of radii, check which one produces a boolean with reasonable variance 
# AND correlates better with rent  ---for (dist_highway_m, dist_mall_m, dist_river_m, dist_mountain_m)
for radius in [250, 500, 750, 1000, 1500, 2000]:
    near_hw = (df['dist_river_m'] <= radius).astype(int)
    true_rate = near_hw.mean()
    corr = near_hw.corr(df['rent'])
    print(f"radius={radius}m | True rate={true_rate:.2%} | corr with rent={corr:.4f}")

radius=250m | True rate=11.34% | corr with rent=-0.0868
radius=500m | True rate=30.91% | corr with rent=-0.0816
radius=750m | True rate=54.67% | corr with rent=-0.0701
radius=1000m | True rate=68.90% | corr with rent=-0.0440
radius=1500m | True rate=86.91% | corr with rent=0.0078
radius=2000m | True rate=94.30% | corr with rent=0.0546


In [26]:
# Test a range of radii, check which one produces a boolean with reasonable variance 
# AND correlates better with rent  ---for (dist_highway_m, dist_mall_m, dist_river_m, dist_mountain_m)
for radius in [250, 500, 750, 1000, 1500, 2000]:
    near_hw = (df['dist_mountain_m'] <= radius).astype(int)
    true_rate = near_hw.mean()
    corr = near_hw.corr(df['rent'])
    print(f"radius={radius}m | True rate={true_rate:.2%} | corr with rent={corr:.4f}")

radius=250m | True rate=0.02% | corr with rent=-0.0012
radius=500m | True rate=0.38% | corr with rent=-0.0037
radius=750m | True rate=1.64% | corr with rent=0.0100
radius=1000m | True rate=2.44% | corr with rent=-0.0055
radius=1500m | True rate=5.15% | corr with rent=-0.0053
radius=2000m | True rate=9.90% | corr with rent=-0.0150


In [27]:
df['near_highway'] = (df['dist_highway_m'] <= 250).astype(int)
df['near_mall'] = (df['dist_mall_m'] <= 250).astype(int)
df['near_river'] = (df['dist_river_m'] <= 250).astype(int)
df['near_mountain'] = (df['dist_mountain_m'] <= 2000).astype(int)

In [28]:
#1. Drop the raw distance columns — they did their job (calibrating the thresholds), but your chatbot can only realistically ask yes/no questions, not "how many meters away," so there's no reason to keep them in the modeling dataset:
df = df.drop(columns=['dist_highway_m', 'dist_mall_m', 'dist_river_m', 'dist_mountain_m'])
df.shape

(117375, 14)

In [29]:
#Save the final EDA output, ready for Linear Regression → Random Forest → XGBoost:
df.to_csv('../data/rentora_eda_final.csv', index=False)
print("Saved:", df.shape)

Saved: (117375, 14)


In [30]:
df[df['city'] == 'Hisar'].shape

(0, 14)